In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Inatll all required packages
!pip install biopython
!pip install scispacy
!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.0/en_ner_bionlp13cg_md-0.5.0.tar.gz
!pip -q install pandas requests tqdm
!pip install -q mygene
#!pip install "/content/drive/MyDrive/KG_Ebola/en_ner_bionlp13cg_md-0.5.4.tar.gz"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00


In [ ]:
#Import all packages
from Bio import Entrez
import time
import csv
import re
from collections import defaultdict
from typing import List, Tuple
import spacy

#### Retrieve Pubmed ids for Ebola and save it in a .csv file

Step 1: Find pubmed ids which is open access

In [ ]:
Entrez.email = "koyel.mandal23@gmail.com"

# Ebola-related search terms
search_terms = [
    "Ebola virus",
    "Ebola virus disease",
    "Ebola hemorrhagic fever",
    "Zaire ebolavirus", "ZEBOV", "Zaire Ebola virus",
    "Sudan ebolavirus", "SEBOV", "Sudan Ebola virus",
    "Bundibugyo ebolavirus",
    "Taï Forest ebolavirus", "TAFV",
    "EVD", "EBOV",
]

# Combine terms into a PubMed query
query = " OR ".join(f'"{term}"[Title/Abstract]' for term in search_terms)

# Parameters for batching
retmax = 1000
retstart = 0
all_pmids = []
print("Starting PubMed search...")

# Loop to fetch all PMIDs
while True:
    handle = Entrez.esearch(db="pubmed", term=query, retmax=retmax, retstart=retstart, sort="relevance")
    record = Entrez.read(handle)
    handle.close()

    pmids = record["IdList"]
    all_pmids.extend(pmids)

    print(f"Retrieved {len(pmids)} PMIDs at start={retstart}")

    if len(pmids) < retmax:
        break  # No more results to fetch

    retstart += retmax
    time.sleep(0.5)  # Respect NCBI server rate limits

print(f"\n Total PMIDs collected: {len(all_pmids)}")

# Save to CSV
csv_filename = r"/content/drive/My Drive/KG_Ebola/ebola_pmids.csv"
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["PMID"])
    for pmid in all_pmids:
        writer.writerow([pmid])

print(f"PMIDs saved to '{csv_filename}'")


Starting PubMed search...
Retrieved 1000 PMIDs at start=0
Retrieved 1000 PMIDs at start=1000
Retrieved 1000 PMIDs at start=2000
Retrieved 1000 PMIDs at start=3000
Retrieved 1000 PMIDs at start=4000
Retrieved 1000 PMIDs at start=5000
Retrieved 1000 PMIDs at start=6000
Retrieved 1000 PMIDs at start=7000
Retrieved 626 PMIDs at start=8000

 Total PMIDs collected: 8626
PMIDs saved to '/content/drive/My Drive/KG_Ebola/ebola_pmids.csv'


In [ ]:
print("The total pubmed ids found for Ebola is", len(all_pmids))

The total pubmed ids found for Ebola is 8626


####Since not all PubMed articles are open access, we will only consider those for which the full text of the paper is accessible.

In [ ]:
# --- 1. Load the PMIDs --------------------------------------------------------
input_file = "/content/drive/My Drive/KG_Ebola/ebola_pmids.csv"

with open(input_file, newline='') as f:
    reader = csv.reader(f)
    next(reader)                     # skip header
    all_pmids = [row[0] for row in reader]

# --- 2. Function to fetch PMC IDs -------------------------------------------
def get_pmc_links(pmids):
    results = []
    for i, pmid in enumerate(pmids, start=1):
        try:
            handle = Entrez.elink(
                dbfrom="pubmed",
                db="pmc",
                id=pmid,
                linkname="pubmed_pmc"
            )
            record = Entrez.read(handle)
            handle.close()

            # Is there at least one PMC link?
            if record[0]["LinkSetDb"]:
                pmc_id = record[0]["LinkSetDb"][0]["Link"][0]["Id"]
                fulltext_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}"
                results.append([pmid, "Yes", f"PMC{pmc_id}", fulltext_url])
            else:
                results.append([pmid, "No", "", ""])

        except Exception as e:
            print(f"Error on PMID {pmid}: {e}")
            results.append([pmid, "Error", "", ""])

        # polite pause for NCBI servers
        time.sleep(0.3)
        # Uncomment to see progress:
        # print(f"{i}/{len(pmids)} processed")

    return results

# --- 3. Run the fetch and save ----------------------------------------------
pmc_results = get_pmc_links(all_pmids)

output_file = "/content/drive/My Drive/KG_Ebola/ebola_pmids_with_links.csv"
with open(output_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["PMID", "OpenAccess", "PMC_ID", "FullTextURL"])
    writer.writerows(pmc_results)

print(f"\n Full‑text link table saved to: {output_file}")



 Full‑text link table saved to: /content/drive/My Drive/KG_Ebola/ebola_pmids_with_links.csv


#### Next filter the ebola_pmids_with_links.csv file where i will only keep the open access pdf.

In [ ]:
input_file = r"/content/drive/My Drive/KG_Ebola/ebola_pmids_with_links.csv"
output_file = r"/content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv"

with open(input_file, newline='') as infile, open(output_file, mode='w', newline='') as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    header = next(reader)
    writer.writerow(header)  # write header to new file

    for row in reader:
        if row[1].strip() == "Yes":
            writer.writerow(row)

print(f" Closed-access PMIDs saved to: {output_file}")


 Closed-access PMIDs saved to: /content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv


#### Step2: Using PubMed IDs, identify Ebola virus proteins GP (glycoprotein), NP (nucleoprotein), L (RNA-dependent RNA polymerase), (matrix protein) VP24, VP30, VP35, and VP40 , and extract the host proteins reported to interact with them.
1.   Node type
*   Viral protein, Host protein, Gene, Drugs, Pathways

2.   Relationship types
*   interacts_with, inhibits, associates, activates, participates_in, has_function, involved in, located in, target of, associated with, encodes, binds.

### Triplets from pubmed (Sujoy)

In [ ]:
# Sujoy
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import spacy
from nltk.stem import PorterStemmer

# Load SpaCy model and stemmer
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()

# Define Ebola proteins and interaction verbs
ebola_proteins = [
    "GP", "glycoprotein", "NP", "nucleoprotein", "L", "RNA-dependent RNA polymerase",
    "VP24", "VP30", "VP35", "VP40"
]

interaction_verbs = [
    "bind", "interact", "associate", "activate", "inhibit", "stimulate", "block",
    "recruit", "modulate", "induce", "trigger", "suppress", "phosphorylate", "dephosphorylate",
    "cleave", "express", "stabilize", "destabilize", "prevent", "enhance", "mediate",
    "degrade", "form", "inactivate", "ubiquitinate", "acetylate", "translocate", "transport",
    "recognize", "neutralize", "attach", "penetrate", "encode", "interfere", "perturb",
    "aggregate", "affect", "alter", "change", "connect", "control", "convert", "cooperate",
    "crosslink", "disrupt", "enable", "facilitate", "impair", "impact", "involve", "join",
    "link", "maintain", "promote", "regulate", "replace", "respond", "sequester", "signal",
    "switch", "target", "translate", "transmit", "transform", "utilize"
]

# --- Functions ---

def fetch_pmc_xml(pmc_id):
    url = f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{pmc_id[3:]}&metadataPrefix=pmc"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        print(f"❌ Failed to fetch PMC article: {pmc_id}")
        return None

def extract_text_from_xml(xml_data):
    soup = BeautifulSoup(xml_data, "xml")
    body = soup.find("body")
    return body.get_text(separator=" ") if body else ""

def find_interactions(text, protein_list, pmc_id):
    sentences = re.split(r'\.\s+', text)
    results = []

    for sentence in sentences:
        doc = nlp(sentence)
        lemmas = [token.lemma_.lower() for token in doc]
        stems = [stemmer.stem(token.text.lower()) for token in doc]

        for ebola_protein in protein_list:
            if re.search(rf"\b{re.escape(ebola_protein)}\b", sentence, re.IGNORECASE):
                for verb in interaction_verbs:
                    if verb in lemmas or stemmer.stem(verb) in stems:
                        # Try extracting host protein using noun chunks
                        host_candidates = [
                            chunk.text.strip() for chunk in doc.noun_chunks
                            if chunk.text.lower() not in [p.lower() for p in protein_list]
                            and not re.search(r"\b(ebola|virus|viral)\b", chunk.text.lower())
                            and 1 < len(chunk.text.strip()) < 50
                        ]

                        # Fallback: use proper nouns or capitalized words
                        if not host_candidates:
                            host_candidates = [
                                token.text for token in doc
                                if token.pos_ in ["PROPN", "NOUN"]
                                and token.text.lower() not in [p.lower() for p in protein_list]
                                and token.text[0].isupper()
                            ]

                        host_protein = host_candidates[0] if host_candidates else "Unknown"

                        results.append({
                            "PMCID": pmc_id,
                            "Ebola Protein": ebola_protein,
                            "Interaction": verb,
                            "Host Protein": host_protein
                        })
                        break
    return results

# --- Main pipeline ---

def extract_interactions_from_file(file_path):
    # Step 1: Load CSV
    #df_input = pd.read_csv("/content/subset_ebola_pmids_open_access.csv")
    #df_input = pd.read_csv("/content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv")
    df_input = pd.read_csv(file_path)
    # Step 2: Extract PMC IDs from 3rd column (index = 2)
    pmc_ids = df_input.iloc[:, 2].dropna().astype(str).unique()

    # Step 3: Process each PMC ID
    all_results = []
    for pmc_id in pmc_ids:
        print(f"Processing {pmc_id}...")
        xml_data = fetch_pmc_xml(pmc_id)
        if xml_data:
            text_data = extract_text_from_xml(xml_data)
            results = find_interactions(text_data, ebola_proteins, pmc_id)
            all_results.extend(results)

    # Step 4: Return as DataFrame
    return pd.DataFrame(all_results, columns=["PMCID", "Ebola Protein", "Interaction", "Host Protein"])


# --- Example usage ---
result_df = extract_interactions_from_file(r"/content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv")
print(result_df.head())
result_df.to_csv(r"/content/drive/My Drive/KG_Ebola/ebola_host_interactions.csv", index=False)


Triplets from full biomedical NER using pubmed id and two databases

In [ ]:
MODEL_TAR="/content/drive/MyDrive/KG_Ebola/en_ner_bionlp13cg_md-0.5.4.tar.gz"
!pip install "$MODEL_TAR"

Processing ./drive/MyDrive/KG_Ebola/en_ner_bionlp13cg_md-0.5.4.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for en_ner_bionlp13cg_md: filename=en_ner_bionlp13cg_md-0.5.4-py3-none-any.whl size=119814670 sha256=9f48ceec06d212d58bbe39e1fcf12ddb98407ae584bbe562fe2054c1bb7ffcc7
  Stored in directory: /root/.cache/pip/wheels/f9/e1/19/ea4c68df095a4a87a9856e505c83d88de6bac00f3d62acc62e
Successfully built en_ner_bionlp13cg_md
  Attempting uninstall: en_ner_bionlp13cg_md
    Found existing installation: en_ner_bionlp13cg_md 0.5.4
    Uninstalling en_ner_bionlp13cg_md-0.5.4:
      Successfully uninstalled en_ner_bionlp13cg_md-0.5.4


In [ ]:
from pathlib import Path
import re
import requests
import pandas as pd
import spacy, scispacy  # noqa: F401 – keep import for model availability
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer

# ──────────────────────────────────────────────────────────────────────────
# 1.  Load SciSpacy model & basic helpers
# ──────────────────────────────────────────────────────────────────────────
nlp = spacy.load("en_ner_bionlp13cg_md")  # full biomedical NER & parser
stemmer = PorterStemmer()

# ──────────────────────────────────────────────────────────────────────────
# 1‑A. Ebola protein synonyms → canonical mapping
#      • Keys   = canonical names we want to appear in the output
#      • Values = every spelling found in text (lower‑case)
# ──────────────────────────────────────────────────────────────────────────
EBOLA_SYNONYMS: dict[str, set[str]] = {
    "GP": {"gp", "glycoprotein", "glycoproteins"},
    "NP": {"np", "nucleoprotein", "nucleoproteins"},
    "L": {"l", "rna‑dependent rna polymerase", "rdrp"},
    "VP24": {"vp24"},
    "VP30": {"vp30"},
    "VP35": {"vp35"},
    "VP40": {"vp40"},
}

# Flatten to lookup tables
syn2canon = {syn: canon for canon, syns in EBOLA_SYNONYMS.items() for syn in syns}
# Set of every synonym in lower‑case for quick membership tests
ebola_tokens_lower = set(syn2canon)

def canonical_protein(token: str) -> str:
    """Return canonical Ebola protein name for any synonym, else 'Unknown'."""
    return syn2canon.get(token.lower(), "Unknown")

# ──────────────────────────────────────────────────────────────────────────
# 1‑B. Verb list that signals a molecular interaction
#      Feel free to extend if you need extra verbs later.
# ──────────────────────────────────────────────────────────────────────────
interaction_verbs = [
    "bind", "interact", "associate", "activate", "inhibit", "stimulate",
    "block", "recruit", "modulate", "induce", "trigger", "suppress",
    "phosphorylate", "dephosphorylate", "cleave", "express", "stabilize",
    "destabilize", "prevent", "enhance", "mediate", "degrade", "form",
    "inactivate", "ubiquitinate", "acetylate", "translocate", "transport",
    "recognize", "neutralize", "attach", "penetrate", "encode", "interfere",
    "perturb", "aggregate", "affect", "alter", "change", "connect", "control",
    "convert", "cooperate", "crosslink", "disrupt", "enable", "facilitate",
    "impair", "impact", "involve", "join", "link", "maintain", "promote",
    "regulate", "replace", "respond", "sequester", "signal", "switch",
    "target", "translate", "transmit", "transform", "utilize",
]

# ──────────────────────────────────────────────────────────────────────────
# 2.  Helper functions: download → clean → NER
# ──────────────────────────────────────────────────────────────────────────

def fetch_pmc_xml(pmc_id: str) -> str | None:
    """Download full‑text XML from PMC given a PMCID (e.g. 'PMC1234567')."""
    url = (
        "https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord"
        f"&identifier=oai:pubmedcentral.nih.gov:{pmc_id[3:]}&metadataPrefix=pmc"
    )
    r = requests.get(url, timeout=30)
    return r.text if r.status_code == 200 else None


def extract_body_text(xml: str) -> str:
    """Extract plain text of the <body> tag from PMC XML."""
    soup = BeautifulSoup(xml, "xml")
    body = soup.find("body")
    return body.get_text(" ") if body else ""


def candidate_hosts(doc) -> list[str]:
    """Return list of non‑Ebola proteins/genes mentioned in a sentence."""
    # 1) Named‑entity strings (preferred, highest precision)
    ents = [
        ent.text.strip()
        for ent in doc.ents
        if ent.label_ in {"GENE_OR_GENE_PRODUCT", "PROTEIN"}
        and ent.text.lower() not in ebola_tokens_lower
    ]
    if ents:
        return ents

    # 2) Short ALLCAPS terms (fallback, e.g. 'STAT1')
    caps = re.findall(r"\b[A-Z0-9\-]{2,10}\b", doc.text)
    caps = [c for c in caps if c.lower() not in ebola_tokens_lower]
    if caps:
        return caps

    # 3) Finally, noun chunks minus Ebola words (lowest precision)
    chunks = [
        ch.text.strip()
        for ch in doc.noun_chunks
        if ch.text.lower() not in ebola_tokens_lower
        and not re.search(r"\b(ebola|virus|viral)\b", ch.text.lower())
    ]
    return chunks or ["Unknown"]


# ──────────────────────────────────────────────────────────────────────────
# 3.  Sentence‑level interaction mining
# ──────────────────────────────────────────────────────────────────────────

def find_interactions(text: str, pmc_id: str) -> list[dict]:
    """Return a list of interaction dicts mined from article body text."""
    hits: list[dict] = []
    for sent in nlp(text).sents:
        doc = nlp(sent.text)

        lemmas = {t.lemma_.lower() for t in doc}
        stems = {stemmer.stem(t.lower_) for t in doc}  # type: ignore[attr-defined]

        # 3‑A. Check if any Ebola synonym is in this sentence
        match = next(
            (
                tok
                for tok in ebola_tokens_lower
                if re.search(rf"\b{re.escape(tok)}\b", sent.text, re.I)
            ),
            None,
        )
        if not match:
            continue  # skip sentences with no Ebola protein mention

        # 3‑B. At least one interaction verb must be present
        verbs = [v for v in interaction_verbs if v in lemmas or stemmer.stem(v) in stems]
        if not verbs:
            continue

        # 3‑C. Collect host proteins/genes and emit rows
        ebola_hit = canonical_protein(match)
        hosts = candidate_hosts(doc)

        for host in hosts:
            for v in verbs:
                hits.append(
                    {
                        "PMCID": pmc_id,
                        "Ebola Protein": ebola_hit,
                        "Interaction": v,
                        "Host Protein": host,
                    }
                )
    return hits


# ──────────────────────────────────────────────────────────────────────────
# 4.  Main pipeline: iterate through articles & dump CSV
# ──────────────────────────────────────────────────────────────────────────

def extract_interactions_from_file(csv_path: str | Path, max_articles: int | None = 100) -> pd.DataFrame:
    """Read a CSV that has PMCIDs (col‑2) and mine interactions."""
    df_in = pd.read_csv(csv_path)
    pmcids = df_in.iloc[:, 2].dropna().astype(str).unique()
    if max_articles:
        pmcids = pmcids[:max_articles]

    all_rows: list[dict] = []
    for pmc in pmcids:
        print(f"Processing {pmc} …")
        xml = fetch_pmc_xml(pmc)
        if not xml:
            print("  ⚠️  could not download XML")
            continue
        text = extract_body_text(xml)
        all_rows.extend(find_interactions(text, pmc))

    return pd.DataFrame(
        all_rows,
        columns=["PMCID", "Ebola Protein", "Interaction", "Host Protein"],
    )


# ──────────────────────────────────────────────────────────────────────────
# 5.  Example usage (stand‑alone script)
# ──────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    input_csv = "/content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv"
    output_csv = "/content/drive/My Drive/KG_Ebola/ebola_host_interactions_ner.csv"

    # Set max_articles=None to scan the entire list
    #df = extract_interactions_from_file(input_csv, max_articles=5)  # e.g. max_articles=50
    df = extract_interactions_from_file(input_csv, max_articles=None)
    print(df.head())
    df.to_csv(output_csv, index=False)
    print("✅ Results saved →", output_csv)


In [ ]:
#from pathlib import Path
#import pandas as pd

#UNIPROT_TSV = "/content/drive/My Drive/KG_Ebola/uniprotkb_organism_id_9606_AND_reviewed_2025_06_16.tsv"
#SYMBOL_TXT  = "/content/drive/My Drive/KG_Ebola/human_uniprot_symbols.txt"

#df_u = pd.read_csv(UNIPROT_TSV, sep="\t", low_memory=False)

# Choose the correct column name automatically
#col = "Gene Names" if "Gene Names" in df_u.columns else "Gene names (primary)"

#primary_syms = (
#    df_u[col]
 #     .dropna()
  #    .str.split()          # split on whitespace
   #   .str[0]               # first token = primary symbol
    #  .str.strip()          # remove any stray whitespace
      #.str.upper()          # standardise to upper‑case
     # .unique()
#)

# 🔑  Write one symbol per line with a final trailing newline
#with open(SYMBOL_TXT, "w", encoding="utf-8", newline="\n") as out:
 #   out.write("\n".join(primary_syms) + "\n")

#print(f"✅ Wrote {len(primary_syms):,} unique symbols → {SYMBOL_TXT}")


Process the file to keep host proteins matched with uniprot protein/gene names. we kept genes

In [ ]:
import re, pandas as pd
from pathlib import Path

# ──────────────────────────────
# 1.  Paths  ── edit if needed
# ──────────────────────────────
RAW_XLSX    = "/content/drive/My Drive/KG_Ebola/ebola_host_interactions.csv"
CLEAN_CSV   = "/content/drive/My Drive/KG_Ebola/ebola_host_interactions_clean.csv"
UNIPROT_TSV = "/content/drive/My Drive/KG_Ebola/uniprotkb_organism_id_9606_AND_reviewed_2025_06_16.tsv"

# ──────────────────────────────
# 2.  Reusable constants
# ──────────────────────────────
PPI_VERBS = {        # keep as‑is
    "bind","interact","associate","activate","inhibit","stimulate","block",
    "recruit","modulate","induce","trigger","suppress","phosphorylate",
    "dephosphorylate","cleave","express","stabilize","destabilize","prevent",
    "enhance","mediate","degrade","form","inactivate","ubiquitinate",
    "acetylate","translocate","transport","recognize","neutralize","attach",
    "penetrate","encode","interfere","perturb","aggregate","affect","alter",
    "change","connect","control","convert","cooperate","crosslink","disrupt",
    "enable","facilitate","impair","impact","involve","join","link","maintain",
    "promote","regulate","replace","respond","sequester","signal","switch",
    "target","translate","transmit","transform","utilize"
}

EBOLA_SYNONYMS = {
    "gp","glycoprotein","np","nucleoprotein","l",
    "vp24","vp30","vp35","vp40","rdrp"
}

BAN_WORDS = {
    "virus","viral","ebola","infection","protein","proteins",
    "cell","cells","domain","motif","gp","glycoprotein"
}

SYMBOL_RE = re.compile(r"^[A-Z0-9\-]{2,10}$")   # (removed stray backslash)

# ──────────────────────────────
# 3.  Build UniProt symbol set
# ──────────────────────────────
print("Loading UniProt TSV …")
u_df = pd.read_csv(UNIPROT_TSV, sep="\t", low_memory=False)

col = "Gene Names" if "Gene Names" in u_df.columns else "Gene names (primary)"
UNIPROT_SET = {
    row.split()[0].strip().upper()
    for row in u_df[col].dropna()
}
print(f"  → collected {len(UNIPROT_SET):,} primary gene symbols")

# ──────────────────────────────
# 4.  Read raw PPI file
# ──────────────────────────────
print("Reading raw PPI file …")
#ppi_df = pd.read_excel(RAW_XLSX)        # (or pd.read_csv if CSV)
ppi_df = pd.read_csv(RAW_XLSX)
print(f"  → {len(ppi_df):,} rows loaded")

#ppi_df["Host Protein"] = ppi_df["Host Protein"].str.strip().str.upper()

#ppi_df["Interaction"]  = ppi_df["Interaction"].str.strip().str.lower()

# ──────────────────────────────
# 5.  Helper to validate host protein
# ──────────────────────────────
def good_host(text: str) -> bool:
    txt = str(text).strip().upper()
    low = txt.lower()

    if low in BAN_WORDS:          # same as before
        return False
    if low in EBOLA_SYNONYMS:     # ← exact match, no substring
        return False
    if txt not in UNIPROT_SET:    # same as before
        return False
    return bool(SYMBOL_RE.fullmatch(txt))
# ──────────────────────────────
# 6.  Apply filters
# ──────────────────────────────
mask = (
    #ppi_df["Interaction"].str.lower().isin(PPI_VERBS) &
    ppi_df["Host Protein"].apply(good_host)
)

clean_df = ppi_df[mask].reset_index(drop=True)
print(f"✅ Filtered down to {len(clean_df):,} high‑confidence PPI rows")

# ──────────────────────────────
# 7.  Save result
# ──────────────────────────────
clean_df.to_csv(CLEAN_CSV, index=False)
print("Saved:", CLEAN_CSV)

# (Optional) peek at the first few rows
display(clean_df.head())


Loading UniProt TSV …
  → collected 20,210 primary gene symbols
Reading raw PPI file …
  → 38,376 rows loaded
✅ Filtered down to 234 high‑confidence PPI rows
Saved: /content/drive/My Drive/KG_Ebola/ebola_host_interactions_clean.csv


,PMCID,Ebola Protein,Interaction,Host Protein
0,PMC4320629,glycoprotein,interact,CD44
1,PMC7126370,GP,interact,TLR4
2,PMC4675769,glycoprotein,facilitate,NPC1
3,PMC7802368,glycoprotein,impact,Impact
4,PMC5115602,GP,cleave,furin


In [ ]:
#import re, pathlib
#import pandas as pd

#RAW_CSV   = "/content/drive/My Drive/KG_Ebola/matched_triplets.xlsx" # Change as per your file name also check the file extension
#CLEAN_CSV = "/content/drive/My Drive/KG_Ebola/ebola_host_interactions_PPI_clean_sujoy.csv" #change as per your file name

#UNIPROT_SYMBOL_FILE = "/content/drive/My Drive/KG_Ebola/human_uniprot_symbols.txt"

#PPI_VERBS = {
#    "bind", "interact", "associate", "activate", "inhibit", "stimulate",
#    "block", "recruit", "modulate", "induce", "trigger", "suppress",
#    "phosphorylate", "dephosphorylate", "cleave", "express", "stabilize",
#    "destabilize", "prevent", "enhance", "mediate", "degrade", "form",
#    "inactivate", "ubiquitinate", "acetylate", "translocate", "transport",
#    "recognize", "neutralize", "attach", "penetrate", "encode", "interfere",
#    "perturb", "aggregate", "affect", "alter", "change", "connect", "control",
#    "convert", "cooperate", "crosslink", "disrupt", "enable", "facilitate",
#    "impair", "impact", "involve", "join", "link", "maintain", "promote",
#    "regulate", "replace", "respond", "sequester", "signal", "switch",
#    "target", "translate", "transmit", "transform", "utilize"
#}

#EBOLA_SYNONYMS = {
#    "gp", "glycoprotein", "np", "nucleoprotein", "l",
#    "vp24", "vp30", "vp35", "vp40", "rdrp"
#}


#BAN_WORDS = {
#    "virus", "viral", "ebola", "infection", "protein", "proteins",
#    "cell", "cells", "domain", "motif", "gp", "glycoprotein"
#}

#SYMBOL_RE = re.compile(r"^[A-Z0-9\\-]{2,10}$")


#if UNIPROT_SYMBOL_FILE:
#    print("Loading UniProt/HGNC symbols …")
#    UNIPROT_SET = set()
#    with open(UNIPROT_SYMBOL_FILE, "r", encoding="utf-8") as fh:
#        for line in fh:
#            if not line.strip():
#                continue                      # skip empty lines
#            sym = line.strip().split()[0]     # take the first whitespace‑separated token
#            UNIPROT_SET.add(sym)
#    print(f"  → loaded {len(UNIPROT_SET):,} symbols")
#else:
#    UNIPROT_SET = None


#print("Reading raw file …")
#df = pd.read_csv(RAW_CSV) # if csv
#df = pd.read_excel(RAW_CSV)
#print(f"  → {len(df):,} rows loaded")


#def good_host(text: str) -> bool:
#    txt = str(text).strip()
#    low = txt.lower()
#    if low in BAN_WORDS:
#        return False
#    if any(term in low for term in EBOLA_SYNONYMS):
#        return False
#    if UNIPROT_SET is not None and txt not in UNIPROT_SET:
#        return False
#    return bool(SYMBOL_RE.fullmatch(txt))



#mask = (
#    df["Interaction"].isin(PPI_VERBS) &
#    df["Host Protein"].apply(good_host)
#)

#clean_df = df[mask].reset_index(drop=True)
#print(f" Filtered down to {len(clean_df):,} high‑confidence PPI rows")

#clean_df.to_csv(CLEAN_CSV, index=False)
#print(" Saved:", CLEAN_CSV)
#display(clean_df.head())

Intact database



In [ ]:
# Ebola viral proteins (Just to know)

# Dictionary of Ebola virus proteins and their known aliases
#ebola_protein_aliases = {
#    "GP": ["GP", "glycoprotein", "GP1", "GP2", "envelope glycoprotein"],
#    "NP": ["NP", "nucleoprotein"],
#    "L": ["L", "RNA-dependent RNA polymerase", "RdRp"],
#    "VP24": ["VP24", "viral protein 24"],
#    "VP30": ["VP30", "viral protein 30"],
#    "VP35": ["VP35", "viral protein 35"],
#    "VP40": ["VP40", "matrix protein", "viral matrix protein"]
#}

# Optionally, flatten for fast lookup (e.g., during NER)
#alias_to_standard_name = {}
#for canonical, aliases in ebola_protein_aliases.items():
#    for alias in aliases:
#        alias_to_standard_name[alias.lower()] = canonical

# Example usage
#def normalize_ebola_protein(name):
#    """Normalize any alias to canonical Ebola viral protein name."""
#    return alias_to_standard_name.get(name.lower(), None)

# Demo
#if __name__ == "__main__":
#    test_terms = ["GP", "gp1", "rna-dependent rna polymerase", "matrix protein", "VP30"]
#    for term in test_terms:
#        print(f"{term} → {normalize_ebola_protein(term)}")


GP → GP
gp1 → GP
rna-dependent rna polymerase → L
matrix protein → VP40
VP30 → VP30


### Viral protein to host protein interactions tripltes using IntAct database.

In [ ]:

import requests, pandas as pd, time
from collections import defaultdict


VIRAL_SYMBOLS = ["GP", "NP", "VP24", "VP30", "VP35", "VP40", "L"]

VIRAL_TAXIDS = {
    "taxid:186538",   # Zaire ebolavirus (species)
    "taxid:128952",   # Ebola virus – strain Mayinga‑76
    "taxid:186540",   # Sudan ebolavirus
    "taxid:565995",   # Bundibugyo ebolavirus
    "taxid:186541",   # Taï Forest ebolavirus
}

UNI_URL = "https://rest.uniprot.org/uniprotkb/search"

def get_accessions(symbol: str, taxid: str) -> list[str]:
    """Return reviewed UniProt accessions for `symbol` in `taxid`."""
    query = (
        f'(gene_exact:{symbol} OR protein_name:"{symbol}") '
        f'AND organism_id:{taxid.split(":")[1]} AND reviewed:true'
    )
    params = {
        "query": query,
        "fields": "accession",
        "size": 500,
        "format": "tsv",
    }
    r = requests.get(UNI_URL, params=params, timeout=30)
    r.raise_for_status()
    return [ln.split("\t")[0] for ln in r.text.splitlines()[1:]]

# Build a dict: viral_symbol → list of accessions
symbol_to_accs = defaultdict(list)
for sym in VIRAL_SYMBOLS:
    for tx in VIRAL_TAXIDS:
        symbol_to_accs[sym] += get_accessions(sym, tx)

## Parse IntAct
def extract_symbol(alias_field: str) -> str:
    for token in alias_field.split("|"):
        tl = token.lower()
        if "(display_short)" in tl or "(gene name)" in tl:
            return token.split(":", 1)[1].split("(")[0].upper()
    return alias_field.split(":", 1)[1].split("(", 1)[0].upper()

def infer_relation(det: str, inter: str) -> str:
    det, inter = det.lower(), inter.lower()
    if any(k in det for k in ("binding", "direct interaction")) or "binding" in inter:
        return "binds"
    return "interacts_with"


PSICQUIC_BASE = (
    "https://www.ebi.ac.uk/Tools/webservices/psicquic/"
    "intact/webservices/current/search/query/"
)

triplets = set()

for viral_sym, accessions in symbol_to_accs.items():
    for acc in accessions:
        miql = f"id:{acc}?format=tab25"
        url  = PSICQUIC_BASE + miql

        resp = requests.get(url, timeout=60)
        resp.raise_for_status()

        for line in resp.text.splitlines():
            if not line or line.startswith("#"):
                continue
            fields = line.split("\t")
            if len(fields) < 15:
                continue

            aliasA, aliasB = fields[4], fields[5]
            taxA,  taxB   = fields[9], fields[10]
            det_method, inter_type = fields[6], fields[11]

            # keep only virus–human
            if "taxid:9606" not in (taxA + taxB):
                continue
            if not any(t in (taxA + taxB) for t in VIRAL_TAXIDS):
                continue

            human_sym = extract_symbol(aliasA if "taxid:9606" in taxA else aliasB)
            relation  = infer_relation(det_method, inter_type)
            triplets.add((viral_sym, relation, human_sym))

        time.sleep(0.3)  # polite pause

df = (
    pd.DataFrame(sorted(triplets),
                 columns=["viral_protein", "relation", "human_protein"])
      .sort_values(["viral_protein", "human_protein"])
)
df.to_csv("/content/drive/My Drive/KG_Ebola/ebola_host_triplets_intact.csv", index=False)
print(f"✅ Saved {len(df):,} unique virus–human triplets to 'ebola_host_triplets_intact.csv'")


✅ Saved 93 unique virus–human triplets to 'ebola_host_triplets_intact.csv'


Use virhostnet database. Find the uniprot for all ebola virus types and strains. from Uniprot.

In [ ]:
import requests, pandas as pd, time
from itertools import product

EBOLA_PROTEINS   = ["GP", "NP", "VP24", "VP30", "VP35", "VP40", "L"]
EBOLA_ORGANISMS  = [
    "Zaire ebolavirus",          # species
    "Ebola virus, strain Mayinga-76",  # strain
    "Sudan ebolavirus",
    "Bundibugyo ebolavirus",
    "Taï Forest ebolavirus",
]

# Map the short symbol to a search phrase UniProt understands
SEARCH_TERM = {
    "GP":   "glycoprotein",
    "NP":   "nucleoprotein",
    "VP24": "VP24",
    "VP30": "VP30",
    "VP35": "VP35",
    "VP40": "VP40",
    "L":    "RNA-dependent RNA polymerase",
}


API = "https://rest.uniprot.org/uniprotkb/search"

def query_uniprot(term: str, organism: str):
    query = f'({term}) AND organism_name:"{organism}" AND reviewed:true'
    resp = requests.get(
        API,
        params={
            "query":  query,
            "format": "json",
            "fields": "accession,protein_name,organism_name",
            "size":   500,
        },
        timeout=30,
    )
    resp.raise_for_status()
    for r in resp.json().get("results", []):
        yield (
            r["primaryAccession"],
            r.get("proteinDescription", {})
             .get("recommendedName", {})
             .get("fullName", {})
             .get("value", "N/A"),
            r["organism"]["scientificName"],
        )

rows = []
for symbol, org in product(EBOLA_PROTEINS, EBOLA_ORGANISMS):
    term = SEARCH_TERM[symbol]
    for acc, pname, org_name in query_uniprot(term, org):
        rows.append((symbol, org_name, acc, pname))
    time.sleep(0.2)            # polite pause


df = (
    pd.DataFrame(rows,
                 columns=["Protein_Symbol", "Organism", "UniProt_ID", "Protein_Name"])
      .drop_duplicates()
      .sort_values(["Protein_Symbol", "Organism"])
)
df.to_csv("/content/drive/My Drive/KG_Ebola/ebola_uniprot_ids.csv", index=False)
print(f"✅  Saved {len(df)} rows to 'ebola_uniprot_ids.csv'")


✅  Saved 109 rows to 'ebola_uniprot_ids.csv'


Read ebola_uniprot_ids.csv and retreive all the uniprot ids for virhostnet database.

In [ ]:
import pandas as pd

# 1.  Load the CSV you just wrote
file_path = "/content/drive/My Drive/KG_Ebola/ebola_uniprot_ids.csv"
df = pd.read_csv(file_path)

# 2.  Pull the UniProt_ID column and make it unique
uniprot_ids = df["UniProt_ID"].drop_duplicates().tolist()

print(f"{len(uniprot_ids)} unique UniProt IDs loaded.")
print(uniprot_ids[:10])   # peek at the first few


42 unique UniProt IDs loaded.
['Q66814', 'P60172', 'Q7T9D9', 'Q7T9E0', 'P0C772', 'Q66798', 'P60173', 'Q66810', 'Q66811', 'P87671']


Triplets using virhostnet

In [ ]:
import requests
import pandas as pd
import time


def fetch_ppi(uniprot_id):
    url = f"http://virhostnet.prabi.fr:9090/psicquic/webservices/current/search/query/uniprotkb:{uniprot_id}?format=tab25"
    try:
        response = requests.get(url, timeout=60)
        if response.status_code != 200 or not response.text.strip():
            return []
        interactions = []
        for line in response.text.strip().split("\n"):
            f = line.split("\t")
            if len(f) < 15 or "taxid:9606" not in (f[9]+f[10]):
                continue
            host = f[0].split(":")[1]
            viral  = f[1].split(":")[1]
            relation = f[11].split("(")[1].rstrip(")") if "(" in f[11] else f[11]
            pubmed = f[8] if "pubmed" in f[8] else "-"
            interactions.append((viral, relation, host, pubmed))
        return interactions
    except Exception as e:
        print(f"Failed for {uniprot_id}: {e}")
        return []

# Gather all interactions
all_rows = []
for uid in uniprot_ids:
    all_rows.extend(fetch_ppi(uid))
    time.sleep(0.3)  # polite request interval

# Save to CSV
df = pd.DataFrame(all_rows, columns=["Viral_Protein", "Relation", "Host_Protein", "PubMed_ID"])
df.drop_duplicates().to_csv("/content/drive/My Drive/KG_Ebola/ebola_host_triplets_virhostnet.csv", index=False)
print("Saved to ebola_host_triplets_virhostnet.csv")


Saved to ebola_ebola_host_triplets_virhostnet.csv


Map all uniprot to protein names

In [ ]:
import pandas as pd, requests, mygene, time, re

ppi_path = "/content/drive/My Drive/KG_Ebola/ebola_host_triplets_virhostnet.csv"
out_path = "/content/drive/My Drive/KG_Ebola/ebola_host_triplets_virhostnet_map.csv"

ppi = pd.read_csv(ppi_path)

viral_ids = ppi["Viral_Protein"].unique().tolist()

def classify_name(name: str) -> str:
    """Map UniProt recommendedName to GP / NP / VP24 … L"""
    name = name.lower()
    if   "glycoprotein"      in name: return "GP"
    elif "nucleoprotein"     in name: return "NP"
    elif re.search(r"\bvp24\b", name): return "VP24"
    elif re.search(r"\bvp30\b", name): return "VP30"
    elif re.search(r"\bvp35\b", name): return "VP35"
    elif re.search(r"\bvp40\b", name): return "VP40"
    elif "rna-dependent" in name or name.startswith("polymerase"): return "L"
    else: return "UNKNOWN"

u2viral = {}                       # UniProt -> GP/NP/…
API = "https://rest.uniprot.org/uniprotkb/"

for uid in viral_ids:
    r = requests.get(f"{API}{uid}.json", timeout=20)
    if r.ok:
        rec = r.json()
        pname = (
            rec["proteinDescription"]
               ["recommendedName"]
               ["fullName"]["value"]
        )
        u2viral[uid] = classify_name(pname)
    else:
        u2viral[uid] = "UNKNOWN"
    time.sleep(0.1)                # courtesy pause


mg = mygene.MyGeneInfo()
host_ids = ppi["Host_Protein"].unique().tolist()

host2gene = {}
for batch in [host_ids[i:i+1000] for i in range(0, len(host_ids), 1000)]:
    hits = mg.querymany(
        batch, scopes="uniprot", fields="symbol", species="human", as_dataframe=False
    )
    for h in hits:
        host2gene[h["query"]] = h.get("symbol", h["query"])
for acc in host_ids:
    host2gene.setdefault(acc, acc)

viral_set = set(u2viral)

def fix_swap(row):
    if row["Viral_Protein"] not in viral_set and row["Host_Protein"] in viral_set:
        row["Viral_Protein"], row["Host_Protein"] = row["Host_Protein"], row["Viral_Protein"]
    return row

ppi = ppi.apply(fix_swap, axis=1)


ppi["Viral_Symbol"] = ppi["Viral_Protein"].map(u2viral).fillna("UNKNOWN")
ppi["Host_Symbol"]  = ppi["Host_Protein"].map(host2gene)

cols_out = ["Viral_Symbol", "Relation", "Host_Symbol","PubMed_ID"]
temp_df = ppi[cols_out].copy()

temp_df = temp_df.drop(columns=["PubMed_ID"])


temp_df = temp_df.rename(columns={
    "Viral_Symbol": "viral_protein",
    "Relation": "relation",
    "Host_Symbol": "human_protein"
})
triplets = temp_df.drop_duplicates()

triplets.to_csv(out_path, index=False)

print(f" {len(triplets)} unique Ebola–human triplets written to → {out_path}")
print("\n Quick peek:\n")
print(triplets.head(8))


INFO:biothings.client:querying 1-219 ...
INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.


 221 unique Ebola–human triplets written to → /content/drive/My Drive/KG_Ebola/ebola_host_triplets_virhostnet_map.csv

 Quick peek:

  viral_protein              relation human_protein
0            GP  physical association         REEP6
1            GP  physical association          PHB2
2            GP  physical association          FUT8
3            GP  physical association          TMX1
4            GP  physical association         GANAB
5            GP  physical association          PON2
6            GP  physical association       GPRASP1
7            GP  physical association          RIF1


First merge all ppi triplets from pubmed. Then remove pumed id. Then again merge with ppi virhostnet and intact. Remove duplicates

In [ ]:
file1 = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/ebola_host_interactions_PPI_clean_ner.csv")
#print(file1.head())
print(file1.shape)
file2 = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/ebola_host_interactions_clean.csv")
print(file2.head())
print(file2.shape)

#Removed pubmedid column, renamd other columns
merged1 = (
    pd.concat([file1, file2], ignore_index=True)
      .iloc[:, 1:]
      .rename(
          columns={
              "Ebola Protein": "Ebola_protein",
              "Interaction":   "Relation",
              "Host Protein":  "Human_protein"

          }
      )
)

print(merged1.head())
print(f"merged shape: {merged1.shape}")


# ── 3. Save
#OUT_CSV = "/content/drive/My Drive/KG_Ebola/ebola_host_interactions_merged.csv"
#merged.to_csv(OUT_CSV, index=False)
#print("Saved →", OUT_CSV)

(11327, 4)
        PMCID Ebola Protein Interaction Host Protein
0  PMC4320629  glycoprotein    interact         CD44
1  PMC7126370            GP    interact         TLR4
2  PMC4675769  glycoprotein  facilitate         NPC1
3  PMC7802368  glycoprotein      impact       Impact
4  PMC5115602            GP      cleave        furin
(234, 4)
  Ebola_protein   Relation Human_protein
0            GP       bind          NPC1
1            GP  transport          NPC1
2            GP     attach          NPC1
3          VP24   activate         STAT1
4          VP24    inhibit         STAT1
merged shape: (11561, 3)


In [ ]:
file3 = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/ebola_host_triplets_intact.csv")
#print(file3.head())
print(file3.shape)
file4 = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/ebola_host_triplets_virhostnet_map.csv")
#print(file4.head())
print(file4.shape)

merged2 = (
    pd.concat([file3, file4], ignore_index=True)
      .rename(
          columns={
              "viral_protein": "Ebola_protein",
              "relation":   "Relation",
              "human_protein":  "Human_protein"

          }
      )
)

print(merged2.head())

print(f"✅ merged shape: {merged2.shape}")

(93, 3)
(221, 3)
  Ebola_protein        Relation Human_protein
0            NP  interacts_with         AP2M1
1            NP  interacts_with          GGA3
2          VP24  interacts_with        ANP32A
3          VP24  interacts_with        ANP32B
4          VP24  interacts_with         CCAR2
✅ merged shape: (314, 3)


In [ ]:
#Now merge both the above files
merged = (
    pd.concat([merged1, merged2], ignore_index=True)
      .drop_duplicates()
)
print(merged.head())

print(f"✅ merged shape: {merged.shape}")


  Ebola_protein   Relation Human_protein
0            GP       bind          NPC1
1            GP  transport          NPC1
2            GP     attach          NPC1
3          VP24   activate         STAT1
4          VP24    inhibit         STAT1
✅ merged shape: (3812, 3)


In [ ]:
synonym_map = {
    "glycoprotein": "GP",
    "gp": "GP",
    "Gp": "GP",
    "L": "L",
    "NP": "NP",
    "VP24": "VP24",
    "VP30": "VP30",
    "VP35": "VP35",
    "VP40": "VP40"
}

merged["Ebola_protein"] = (
    merged["Ebola_protein"]
    .str.strip()
    .str.lower()
    .map(synonym_map)
    .fillna(merged["Ebola_protein"])  # if not in map, keep original
)

merged["Human_protein"] = merged["Human_protein"].str.upper()

# 4. View results
print(merged.head())
print(merged.shape)

OUT_CSV = "/content/drive/My Drive/KG_Ebola/ppi.csv" ##This is the final PPI
merged.to_csv(OUT_CSV, index=False)
print("Saved →", OUT_CSV)

  Ebola_protein   Relation Human_protein
0            GP       bind          NPC1
1            GP  transport          NPC1
2            GP     attach          NPC1
3          VP24   activate         STAT1
4          VP24    inhibit         STAT1
(3812, 3)


Add drug nodes.

In [ ]:
!pip install lxml tqdm

### Dont run

In [ ]:
#Process drugbank.xml file to get required columns in a .csv file
#This is master file, DrugBankID,	Name,	Type,	Groups,	Targets,	Pathways, and	Synonyms. These are separted by |.

from pathlib import Path
import csv
import lxml.etree as ET
from collections import deque
from tqdm import tqdm

# ── CONFIG ─────────────────────────────────────────────────────────────
XML_PATH = Path(r"/content/drive/My Drive/KG_Ebola/full database.xml")          # <── change if needed
OUT_CSV  = Path(r"/content/drive/My Drive/KG_Ebola/drugbank_list.csv")
KEEP_MAX_SYNONYMS = None                        # None = keep all, if want can set the number

# ── 1. Discover namespace once ─────────────────────────────────────────
for _, root in ET.iterparse(XML_PATH, events=("start",)):
    NS_URI = root.tag.split("}")[0].strip("{")
    break
NS = {"db": NS_URI}
TAG_DRUG = f"{{{NS_URI}}}drug"

# ── 2. Count drugs quickly (progress bar needs total) ──────────────────
total = sum(1 for _ in ET.iterparse(XML_PATH, events=("end",), tag=TAG_DRUG))
print(f"Namespace: {NS_URI}\nTotal <drug> entries: {total:,}")

# ── 3. Stream‑parse and write CSV ──────────────────────────────────────
with OUT_CSV.open("w", newline="", encoding="utf‑8") as fh:
    wr = csv.writer(fh)
    wr.writerow([
        "DrugBankID", "Name", "Type", "Groups",
        "Targets", "Pathways", "Synonyms"
    ])

    context = ET.iterparse(XML_PATH, events=("end",), tag=TAG_DRUG)
    for _, drug in tqdm(context, total=total, desc="Extracting"):
        # ── Basic fields ────────────────────────────────────────────
        dbid_tag = drug.find('db:drugbank-id[@primary="true"]', NS)
        dbid = dbid_tag.text.strip() if dbid_tag is not None and dbid_tag.text else ""

        name_tag = drug.find("db:name", NS)
        name = name_tag.text.strip() if name_tag is not None and name_tag.text else ""

        d_type = drug.attrib.get("type", "")

        groups = [g.text for g in drug.findall("db:groups/db:group", NS) if g.text]
        groups_str = "|".join(groups)

        # ── Targets (gene names preferred) ─────────────────────────
        tgt_set = set()
        for tgt in drug.findall("db:targets/db:target", NS):
            gene = tgt.find("db:gene-name", NS)
            if gene is not None and gene.text:
                tgt_set.add(gene.text.strip())
            else:
                tname = tgt.find("db:name", NS)
                if tname is not None and tname.text:
                    tgt_set.add(tname.text.strip())
        targets_str = "|".join(sorted(tgt_set))

        # ── Pathways (names) ───────────────────────────────────────
        pnames = [
            p.find("db:name", NS).text.strip()
            for p in drug.findall("db:pathways/db:pathway", NS)
            if p.find("db:name", NS) is not None and p.find("db:name", NS).text
        ]
        pathways_str = "|".join(dict.fromkeys(pnames))   # dedupe, keep order

        # ── Synonyms (cap length) ─────────────────────────────────
        syns = deque(maxlen=KEEP_MAX_SYNONYMS) if KEEP_MAX_SYNONYMS else []
        for syn in drug.findall("db:synonyms/db:synonym", NS):
            if syn.text:
                if KEEP_MAX_SYNONYMS:
                    syns.append(syn.text.strip())
                else:
                    syns.append(syn.text.strip())
        syn_str = "|".join(syns)

        # ── Write row ─────────────────────────────────────────────
        wr.writerow([dbid, name, d_type, groups_str,
                     targets_str, pathways_str, syn_str])

        # ── Free memory ───────────────────────────────────────────
        drug.clear()
        while drug.getprevious() is not None:
            del drug.getparent()[0]

print("✅ CSV saved to:", OUT_CSV.resolve())


Namespace: http://www.drugbank.ca
Total <drug> entries: 72,838


Extracting: 100%|██████████| 72838/72838 [01:09<00:00, 1047.34it/s]

✅ CSV saved to: /content/drive/My Drive/KG_Ebola/drugbank_list.csv


Dont run

In [ ]:
#Read csv file and preprocess it
import pandas as pd

in_path  = "/content/drive/My Drive/KG_Ebola/drugbank_list_new.csv"
df = pd.read_csv(in_path)

approved_mask = df["Groups"].str.contains("approved", case=False, na=False)
small_mask    = df["Type"].str.contains("small",    case=False, na=False)

filtered = df[approved_mask & small_mask].copy()


print(filtered.head())
print(f"✅ kept {len(filtered):,} of {len(df):,} total rows")


out_path = "/content/drive/My Drive/KG_Ebola/drugbank_approved_smallmolecule_actions.csv"
filtered.to_csv(out_path, index=False)
print("saved →", out_path)


   DrugBankID          Name            Type                    Groups  \
12    DB00006   Bivalirudin  small molecule  approved|investigational   
13    DB00007    Leuprolide  small molecule  approved|investigational   
26    DB00014     Goserelin  small molecule                  approved   
41    DB00027  Gramicidin D  small molecule                  approved   
55    DB00035  Desmopressin  small molecule                  approved   

                                              Targets  \
12                                        Prothrombin   
13            Gonadotropin-releasing hormone receptor   
26  Gonadotropin-releasing hormone receptor|Lutrop...   
41                                                NaN   
55  Vasopressin V1a receptor|Vasopressin V1b recep...   

                      Pathways  \
12  Bivalirudin Action Pathway   
13                         NaN   
26                         NaN   
41                         NaN   
55                         NaN   

             

Do not run. no interactions found.

In [ ]:
from __future__ import annotations
from pathlib import Path
import re
import requests
import pandas as pd
import spacy, scispacy
from spacy.matcher import PhraseMatcher
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Paths (adjust if needed)
# ─────────────────────────────────────────────────────────────────────────────
DRUG_CSV  = Path("/content/drive/My Drive/KG_Ebola/drugbank_approved_smallmolecule.csv")
INPUT_PMC = Path("/content/drive/My Drive/KG_Ebola/ebola_pmids_open_access.csv")
OUTPUT_CSV = Path("/content/drive/My Drive/KG_Ebola/ebola_drug_interactions_ner.csv")


nlp      = spacy.load("en_ner_bionlp13cg_md")
stemmer = PorterStemmer()

def build_drug_matcher(csv_path: Path, nlp) -> tuple[PhraseMatcher, set[str]]:
    """Build PhraseMatcher + lowercase name‑set from the DrugBank small‑molecule CSV."""
    df = pd.read_csv(csv_path)

    names: set[str] = set(df["Name"].dropna().str.lower().str.strip())

    if "Synonyms" in df.columns:
        for syns in df["Synonyms"].fillna(""):
            for syn in str(syns).split("|"):
                syn = syn.strip()
                if syn:
                    names.add(syn.lower())

    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    matcher.add("FDA_SMALL", [nlp.make_doc(n) for n in names])


    return matcher, names

print("🔄 Building drug whitelist matcher …")
drug_matcher, DRUG_NAME_SET = build_drug_matcher(DRUG_CSV, nlp)
print(f"✅ Loaded {len(DRUG_NAME_SET):,} unique drug names/synonyms")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Ebola protein synonyms → canonical
# ─────────────────────────────────────────────────────────────────────────────
EBOLA_SYNONYMS = {
    "GP":  {"gp", "glycoprotein", "glycoproteins"},
    "NP":  {"np", "nucleoprotein", "nucleoproteins"},
    "L":   {"l", "rna-dependent rna polymerase", "rdrp"},
    "VP24": {"vp24"}, "VP30": {"vp30"}, "VP35": {"vp35"}, "VP40": {"vp40"}
}
syn2canon = {syn: canon for canon, syns in EBOLA_SYNONYMS.items() for syn in syns}
ebola_tokens_lower = set(syn2canon)

def canonical_protein(tok: str) -> str:
    return syn2canon.get(tok.lower(), "Unknown")

interaction_verbs = [
    "bind", "interact", "associate", "activate", "inhibit", "stimulate", "block",
    "recruit", "modulate", "induce", "trigger", "suppress", "phosphorylate", "dephosphorylate",
    "cleave", "express", "stabilize", "destabilize", "prevent", "enhance", "mediate",
    "degrade", "form", "inactivate", "ubiquitinate", "acetylate", "translocate", "transport",
    "recognize", "neutralize", "attach", "penetrate", "encode", "interfere", "perturb",
    "aggregate", "affect", "alter", "change", "connect", "control", "convert", "cooperate",
    "crosslink", "disrupt", "enable", "facilitate", "impair", "impact", "involve", "join",
    "link", "maintain", "promote", "regulate", "replace", "respond", "sequester", "signal",
    "switch", "target", "translate", "transmit", "transform", "utilize"
]


def candidate_drugs(doc: spacy.tokens.Doc) -> list[str]:
    """Return FDA‑approved small‑molecule drugs present in the sentence."""
    drugs, seen = [], set()
    for _, start, end in drug_matcher(doc):
        drug = doc[start:end].text.strip()
        key  = drug.lower()
        if key not in seen:  # already guaranteed in whitelist
            drugs.append(drug)
            seen.add(key)
    return drugs


def fetch_pmc_xml(pmc_id: str) -> str | None:
    url = (
        "https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord"
        f"&identifier=oai:pubmedcentral.nih.gov:{pmc_id[3:]}&metadataPrefix=pmc"
    )
    r = requests.get(url, timeout=30)
    return r.text if r.status_code == 200 else None

def extract_body_text(xml: str) -> str:
    soup = BeautifulSoup(xml, "xml")
    body = soup.find("body")
    return body.get_text(" ") if body else ""


def find_drug_interactions(text: str, pmc_id: str):
    rows = []
    for sent in nlp(text).sents:
        doc = nlp(sent.text)

        # (a) Ebola protein present
        ebola = next((tok for tok in ebola_tokens_lower if re.search(rf"\\b{tok}\\b", sent.text, re.I)), None)
        if not ebola:
            continue

        # (b) Interaction verb present
        lemmas = {t.lemma_.lower() for t in doc}
        stems  = {stemmer.stem(t.lower_) for t in doc}  # type: ignore[attr-defined]
        verbs  = [v for v in interaction_verbs if v in lemmas or stemmer.stem(v) in stems]
        if not verbs:
            continue

        # (c) Drug present (from whitelist)
        drugs = candidate_drugs(doc)
        if not drugs:
            continue

        ebola_prot = canonical_protein(ebola)
        for drug in drugs:
            for verb in verbs:
                rows.append({
                    "PMCID": pmc_id,
                    "Ebola Protein": ebola_prot,
                    "Interaction": verb,
                    "Drug": drug,
                    "Sentence": sent.text.strip()
                })
    return rows

# ─────────────────────────────────────────────────────────────────────────────
# 8. Pipeline driver
# ─────────────────────────────────────────────────────────────────────────────

def extract_evd_drug_interactions(pmc_csv: Path, max_articles: int | None = 100):
    df_ids = pd.read_csv(pmc_csv)
    pmcids = df_ids.iloc[:, 2].dropna().astype(str).unique()
    if max_articles:
        pmcids = pmcids[:max_articles]

    all_rows = []
    for pmc in pmcids:
        print(f"🔎 Processing {pmc} …")
        xml = fetch_pmc_xml(pmc)
        if not xml:
            print("  ⚠️  could not download XML")
            continue
        text = extract_body_text(xml)
        all_rows.extend(find_drug_interactions(text, pmc))

    return pd.DataFrame(all_rows, columns=["PMCID", "Ebola Protein", "Interaction", "Drug", "Sentence"])


if __name__ == "__main__":
    df_out = extract_evd_drug_interactions(INPUT_PMC, max_articles=None)
    df_out.to_csv(OUTPUT_CSV, index=False)
    print("✅ Saved →", OUTPUT_CSV)


🔄 Building drug whitelist matcher …
✅ Loaded 15,853 unique drug names/synonyms
🔎 Processing PMC7223853 …
🔎 Processing PMC4605416 …
🔎 Processing PMC5525473 …
🔎 Processing PMC10759212 …
🔎 Processing PMC5653233 …
🔎 Processing PMC9963029 …
🔎 Processing PMC7223059 …
🔎 Processing PMC6784166 …
🔎 Processing PMC6924885 …
🔎 Processing PMC5175058 …
🔎 Processing PMC7122202 …
🔎 Processing PMC5743218 …
🔎 Processing PMC5988239 …
🔎 Processing PMC5620561 …
🔎 Processing PMC9678804 …
🔎 Processing PMC4286619 …
🔎 Processing PMC7088354 …
🔎 Processing PMC4393611 …
🔎 Processing PMC6791933 …
🔎 Processing PMC5026048 …
🔎 Processing PMC10914067 …
🔎 Processing PMC7166513 …
🔎 Processing PMC6816123 …
🔎 Processing PMC10428198 …
🔎 Processing PMC10469131 …
🔎 Processing PMC7232169 …
🔎 Processing PMC4313106 …
🔎 Processing PMC8051212 …
🔎 Processing PMC4320629 …
🔎 Processing PMC6356631 …
🔎 Processing PMC6691429 …
🔎 Processing PMC5082928 …
🔎 Processing PMC4994351 …
🔎 Processing PMC4440555 …
🔎 Processing PMC5407292 …
🔎 Proce

Find Drug and protein interactions from drugbank.

In [ ]:
from pathlib import Path
import csv
import lxml.etree as ET
# import pandas as pd        # ←‑ only needed later for the PPI filter

# ── FILE LOCATIONS ─────────────────────────────────────────────────────
XML_PATH = Path(r"/content/drive/My Drive/KG_Ebola/full database.xml")
OUT_CSV  = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions.csv")

# If / when you want to filter by PPI genes:
# PPI_PATH = Path(r"/content/drive/My Drive/KG_Ebola/ppi.csv")
# ppi_df   = pd.read_csv(PPI_PATH)
# ppi_genes = set(ppi_df["Human_protein"].astype(str).str.strip().str.upper())

# ── 1. Discover DrugBank namespace once ────────────────────────────────
for _, root in ET.iterparse(XML_PATH, events=("start",)):
    NS_URI = root.tag.split("}")[0].strip("{")
    break
NS        = {"db": NS_URI}
TAG_DRUG  = f"{{{NS_URI}}}drug"

# ── 2. Parse XML & write triples ───────────────────────────────────────
with OUT_CSV.open("w", newline="", encoding="utf‑8") as fh:
    wr = csv.writer(fh)
    wr.writerow(["DrugBankID", "DrugName", "Type", "Groups",
                 "GeneSymbol", "Action"])          # ← one action per row

    context = ET.iterparse(XML_PATH, events=("end",), tag=TAG_DRUG)
    for _, drug in context:
        # ── Drug‑level data ──────────────────────────────────────────
        dbid   = drug.findtext('db:drugbank-id[@primary="true"]', "", NS)
        dname  = drug.findtext('db:name', "", NS)
        d_type = drug.get("type", "")

        groups = [g.text for g in drug.findall("db:groups/db:group", NS) if g.text]
        groups_str = "|".join(groups)

        # ── Target loop ─────────────────────────────────────────────
        for tgt in drug.findall("db:targets/db:target", NS):
            polypep = tgt.find("db:polypeptide", NS)
            if polypep is None:
                continue                                # no protein info
            gene = polypep.findtext("db:gene-name", "", NS).strip()
            if not gene:
                continue

            gene_up = gene.upper()

            # ── (optional) PPI gene filter ───────────────────────
            # if gene_up not in ppi_genes:
            #     continue

            actions = [a.text.strip()
                       for a in tgt.findall("db:actions/db:action", NS)
                       if a is not None and a.text]

            if actions:                                   # explode: one row / action
                for act in actions:
                    wr.writerow([dbid, dname, d_type, groups_str, gene, act])
            else:                                         # keep edge even if no action
                wr.writerow([dbid, dname, d_type, groups_str, gene, ""])

        # ── free memory ───────────────────────────────────────────
        drug.clear()
        while drug.getprevious() is not None:
            del drug.getparent()[0]

print("✅ CSV written to:", OUT_CSV.resolve())


✅ CSV written to: /content/drive/My Drive/KG_Ebola/drug_target_actions.csv


In [ ]:
import pandas as pd

# ── I/O paths ──────────────────────────────────────────
in_path  = "/content/drive/My Drive/KG_Ebola/drug_target_actions.csv"
out_path = "/content/drive/My Drive/KG_Ebola/drug_target_actions_remsmgr.csv"

# ── Load ───────────────────────────────────────────────
df = pd.read_csv(in_path)

# ── 1. keep rows whose Type == "small molecule" (case‑insensitive) ──
mask_type = df["Type"].str.lower().eq("small molecule")

# ── 2. keep rows whose Groups column contains the word "approved" ───
#     (matches "approved", "vet_approved", "investigational|approved", …)
mask_appr = df["Groups"].str.contains(r"\bapproved\b", case=False, na=False)

# ── Filter & save ──────────────────────────────────────
filtered = df[mask_type & mask_appr].copy()
filtered.to_csv(out_path, index=False)

print(f"✅ Kept {len(filtered):,} of {len(df):,} rows → {out_path}")


✅ Kept 8,566 of 18,780 rows → /content/drive/My Drive/KG_Ebola/drug_target_actions_remsmgr.csv


In [ ]:
from pathlib import Path
import pandas as pd

# ── PATHS ──────────────────────────────────────────────────────────────
PPI_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/ppi.csv")
SRC_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_remsmgr.csv")  # after previous “approved + small molecule” step
OUT_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_filtered.csv")

# ── 1. Build a case‑insensitive gene‑symbol set from PPI table ─────────
ppi_df = pd.read_csv(PPI_PATH)

# 👉 change the column name below if your header is different
gene_col = "Human_protein"

ppi_gene_set = (
    ppi_df[gene_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .dropna()
        .unique()
)
ppi_gene_set = set(ppi_gene_set)
print("✅ Unique PPI genes:", len(ppi_gene_set))

# ── 2. Load the previously filtered drug–target–action table ───────────
df = pd.read_csv(SRC_PATH)

# ── 3. Keep only rows whose GeneSymbol appears in the PPI set ──────────
mask_in_ppi = (
    df["GeneSymbol"]
      .astype(str)
      .str.strip()
      .str.upper()
      .isin(ppi_gene_set)
)
final_df = df[mask_in_ppi].copy()

final_df["GeneSymbol"] = (
    final_df["GeneSymbol"]
      .astype(str)
      .str.strip()
      .str.upper()
)

# ── 4. Save the result ─────────────────────────────────────────────────
final_df.to_csv(OUT_PATH, index=False)
print(f"✅ Final table: kept {len(final_df):,} of {len(df):,} rows")
print("→", OUT_PATH)


✅ Unique PPI genes: 797
✅ Final table: kept 748 of 8,566 rows
→ /content/drive/My Drive/KG_Ebola/drug_target_actions_filtered.csv


In [ ]:
from pathlib import Path
import pandas as pd

# ── PATHS ──────────────────────────────────────────────────────────────
PPI_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/ppi.csv")
SRC_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_filtered.csv")  # after previous “approved + small molecule” step
OUT_PATH   = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_final.csv")

# ── 1. Build a case‑insensitive gene‑symbol set from PPI table ─────────
ppi_df = pd.read_csv(PPI_PATH)

# 👉 change the column name below if your header is different
gene_col = "Human_protein"

ppi_gene_set = (
    ppi_df[gene_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .dropna()
        .unique()
)
ppi_gene_set = set(ppi_gene_set)
print("✅ Unique PPI genes:", len(ppi_gene_set))

# ── 2. Load the previously filtered drug–target–action table ───────────
df = pd.read_csv(SRC_PATH)

# ── 3. Keep only rows whose GeneSymbol appears in the PPI set ──────────
mask_in_ppi = (
    df["GeneSymbol"]
      .astype(str)
      .str.strip()
      .str.upper()
      .isin(ppi_gene_set)
)
final_df = df[mask_in_ppi].copy()
print(final_df)

final_df["Action"] = (
    final_df["Action"]
      .replace(r"^\s*$", pd.NA, regex=True)  # blank or spaces → NA
      .replace("nan", pd.NA)                 # literal "nan" string → NA
)

# Step 2: Drop rows where Action is now NA
final_df = final_df.dropna(subset=["Action"]).copy()
print(final_df)

tidy_df = (
    final_df
      .drop(columns=["DrugBankID", "Type", "Groups"])
      .rename(columns={"Action": "Relation"})
      .loc[:, ["DrugName", "Relation", "GeneSymbol"]]
)


tidy_df.to_csv(OUT_PATH , index=False)

print("✅ Tidy table written →", OUT_PATH)
print(tidy_df.head())


✅ Unique PPI genes: 797
    DrugBankID             DrugName            Type  \
0      DB00006          Bivalirudin  small molecule   
1      DB00114  Pyridoxal phosphate  small molecule   
2      DB00114  Pyridoxal phosphate  small molecule   
3      DB00114  Pyridoxal phosphate  small molecule   
4      DB00114  Pyridoxal phosphate  small molecule   
..         ...                  ...             ...   
743    DB16390         Mobocertinib  small molecule   
744    DB16650      Deucravacitinib  small molecule   
745    DB16703          Belumosudil  small molecule   
746    DB16826        Repotrectinib  small molecule   
747    DB17472        Pirtobrutinib  small molecule   

                                     Groups GeneSymbol     Action  
0                  approved|investigational         F2  inhibitor  
1    approved|investigational|nutraceutical       ODC1   cofactor  
2    approved|investigational|nutraceutical        TAT   cofactor  
3    approved|investigational|nutraceutical

Drug pathway relations from DrugBank

In [ ]:
from pathlib import Path
import csv
import lxml.etree as ET
import pandas as pd

# ── File paths ──────────────────────────────────────────────────────────
XML_PATH = Path(r"/content/drive/My Drive/KG_Ebola/full database.xml")
DRUG_LIST_CSV = Path(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_final.csv")
OUT_CSV = Path(r"/content/drive/My Drive/KG_Ebola/drug_pathway_simple.csv")

# ── 1. Load drug names from filtered table ──────────────────────────────
drug_df = pd.read_csv(DRUG_LIST_CSV, usecols=["DrugName"])
drug_name_set = set(drug_df["DrugName"].astype(str).str.strip().unique())

# ── 2. Discover namespace ──────────────────────────────────────────────
for _, root in ET.iterparse(XML_PATH, events=("start",)):
    NS_URI = root.tag.split("}")[0].strip("{")
    break
NS = {"db": NS_URI}
TAG_DRUG = f"{{{NS_URI}}}drug"

# ── 3. Parse XML and extract required info ─────────────────────────────
with OUT_CSV.open("w", newline="", encoding="utf-8") as fh:
    wr = csv.writer(fh)
    wr.writerow(["DrugName", "PathwayCategory", "PathwayName"])

    context = ET.iterparse(XML_PATH, events=("end",), tag=TAG_DRUG)
    for _, drug in context:
        dname = drug.findtext("db:name", "", NS).strip()
        if dname not in drug_name_set:
            drug.clear()
            while drug.getprevious() is not None:
                del drug.getparent()[0]
            continue

        for pw in drug.findall("db:pathways/db:pathway", NS):
            pname    = pw.findtext("db:name", "", NS).strip()
            category = pw.findtext("db:category", "", NS).strip()
            if pname and category:
                wr.writerow([dname, category, pname])

        drug.clear()
        while drug.getprevious() is not None:
            del drug.getparent()[0]

print("✅ Saved:", OUT_CSV.resolve())


✅ Saved: /content/drive/My Drive/KG_Ebola/drug_pathway_simple.csv


Protein pathway using Drugbank

In [ ]:
import lxml.etree as ET, csv
from pathlib import Path

XML_PATH = Path("/content/drive/My Drive/KG_Ebola/full database.xml")
UNIPROT_TSV = Path("/content/drive/My Drive/KG_Ebola/uniprotkb_organism_id_9606_AND_reviewed_2025_06_16.tsv")
OUT = Path("/content/drive/My Drive/KG_Ebola/protein_pathway_edges.csv")

# ─── Load UniProt → GeneName mapping ───
id_to_gene = {}
with UNIPROT_TSV.open("r", encoding="utf-8") as f:
    header = f.readline().strip().split("\t")
    entry_idx = header.index("Entry")
    gene_idx = header.index("Gene Names")

    for line in f:
        cols = line.strip().split("\t")
        if len(cols) <= max(entry_idx, gene_idx):
            continue
        uniprot_id = cols[entry_idx].strip()
        gene_name = cols[gene_idx].strip().split(" ")[0]  # take only primary name
        id_to_gene[uniprot_id] = gene_name

# ─── Parse XML ───
for _, root in ET.iterparse(XML_PATH, events=("start",)):
    NS_URI = root.tag.split("}")[0].strip("{")
    break
NS = {"db": NS_URI}

with OUT.open("w", newline="", encoding="utf-8") as fh:
    wr = csv.writer(fh)
    wr.writerow(["UniProtID", "GeneName", "PathwayName", "PathwayCategory", "SMPDB_ID"])

    for _, pw in ET.iterparse(XML_PATH, events=("end",), tag=f"{{{NS_URI}}}pathway"):
        pname = pw.findtext("db:name", "", NS).strip()
        category = pw.findtext("db:category", "", NS).strip()
        smpdb_id = pw.findtext("db:smpdb-id", "", NS).strip()

        block = pw.find("db:enzymes", NS)
        if block is not None:
            for up in block.findall("db:uniprot-id", NS):
                uniprot = up.text.strip()
                gene_name = id_to_gene.get(uniprot, "")
                wr.writerow([uniprot, gene_name, pname, category, smpdb_id])

        pw.clear()
        while pw.getprevious() is not None:
            del pw.getparent()[0]

print("✅ Output with mapped gene names →", OUT)


✅ Output with mapped gene names → /content/drive/My Drive/KG_Ebola/protein_pathway_edges.csv


In [ ]:
import pandas as pd
from pathlib import Path

# 1. Load the two source files
prot_path = pd.read_csv(Path("/content/drive/My Drive/KG_Ebola/protein_pathway_edges.csv"))
ppi       = pd.read_csv(Path("/content/drive/My Drive/KG_Ebola/ppi.csv"))

# 2. Keep only proteins that appear in the PPI file
mask      = prot_path["GeneName"].isin(ppi["Human_protein"])
filtered  = prot_path.loc[mask].drop(columns=["UniProtID", "SMPDB_ID", "PathwayCategory"])

# 3. Add the relation column
filtered["relation"] = "involved_in(DrugBank)"

# Optional: reorder columns if you like
filtered = filtered[["GeneName", "relation", "PathwayName"]]

# 4. Inspect & save
print(filtered.head())
filtered.to_csv(
    Path("/content/drive/My Drive/KG_Ebola/protein_pathway_edges_filtered.csv"),
    index=False
)


  GeneName               relation               PathwayName
0       F2  involved_in(DrugBank)  Lepirudin Action Pathway
3    KLKB1  involved_in(DrugBank)  Lepirudin Action Pathway
4      F11  involved_in(DrugBank)  Lepirudin Action Pathway
5       F9  involved_in(DrugBank)  Lepirudin Action Pathway
7       F5  involved_in(DrugBank)  Lepirudin Action Pathway


Protein/Gene Pathway relation using reactome

In [ ]:
#PPI to uniprot for later use. saved in gene_uniprot.csv

import pandas as pd, numpy as np, requests, io, time
from pathlib import Path
import mygene

ppi_file          = Path("/content/drive/My Drive/KG_Ebola/ppi.csv")            # ← adjust paths if needed
reactome_tsv_url  = "https://reactome.org/download/current/UniProt2Reactome.txt"
batch_size_kegg   = 10       # polite batch size for KEGG REST
out_file          = Path("/content/drive/My Drive/KG_Ebola/protein_pathway_all_sources.csv")

# 1. Load the PPI list (gene symbols)
ppi = pd.read_csv(ppi_file)
symbols = ppi["Human_protein"].dropna().unique().tolist()
print(f"  Genes in PPI: {len(symbols):,}")

# 2. Map gene symbol → UniProt accession (Swiss‑Prot)
mg = mygene.MyGeneInfo()
query = mg.querymany(
    symbols,
    scopes="symbol",
    fields="uniprot.Swiss-Prot",
    species="human",
    as_dataframe=True
)

def pick_uniprot(x):
    """Return a single Swiss‑Prot ID or NaN."""
    if isinstance(x, list):
        return x[0]
    return x

query["SwissProt"] = query["uniprot.Swiss-Prot"].apply(pick_uniprot)
sym_df = (
    query.reset_index()[["query", "SwissProt"]]
         .dropna()
         .rename(columns={"query": "Gene", "SwissProt": "UniProt"})
)
print(f"  Genes mapped to UniProt: {len(sym_df):,}")

sym_df.to_csv(Path("/content/drive/My Drive/KG_Ebola/gene_uniprot.csv"), index=False)

# Query using mygene
mg = mygene.MyGeneInfo()
results = mg.querymany(symbols, scopes='symbol', fields='entrezgene', species='human')

# Convert to DataFrame
ent_df = pd.DataFrame(results)

# Drop entries where entrezgene is missing or not found
if 'notfound' in ent_df.columns:
    ent_df = ent_df[ent_df['notfound'] != True]
if 'entrezgene' not in ent_df.columns:
    raise ValueError(" 'entrezgene' field not found in any result.")

# Keep only relevant columns
ent_df = ent_df[['query', 'entrezgene']].dropna()
ent_df = ent_df.rename(columns={'query': 'GeneSymbol', 'entrezgene': 'EntrezID'})

# Show final output
print(ent_df)

ent_df.to_csv(Path("/content/drive/My Drive/KG_Ebola/EntrezID.csv"), index=False)


INFO:biothings.client:querying 1-797 ...


🧬  Genes in PPI: 797


INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.
INFO:biothings.client:querying 1-797 ...


🆔  Genes mapped to UniProt: 789


INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.


    GeneSymbol EntrezID
0         NPC1     4864
1        STAT1     6772
2         IRF3     3661
3         TBK1    29110
4          CD4      920
..         ...      ...
795      REXO5    81691
796     TIMM8B    26521
797     TIMM13    26517
798     PARP10    84875
799     TIMM8A     1678

[795 rows x 2 columns]


In [ ]:
import pandas as pd
import requests, io


reactome_url = r"/content/drive/My Drive/KG_Ebola/NCBI2Reactome_All_Levels.txt"

col_names = [
    "EntrezID",      # NCBI Gene ID
    "Reactome_ID",   # e.g. R‑HSA‑109582
    "URL",           # Pathway browser link
    "PathwayName",   # Human‑readable pathway name
    "Evidence",      # TAS / IEA / …
    "Species"        # e.g. Homo sapiens
]

reactome_df = pd.read_csv(reactome_url, sep="\t", names=col_names, dtype=str)

reactome_hs = reactome_df[reactome_df["Species"] == "Homo sapiens"]

merged = (
    ent_df.merge(reactome_hs, on="EntrezID", how="inner")
          .loc[:, ["GeneSymbol", "EntrezID", "Reactome_ID", "PathwayName",
                   "Evidence", "URL"]]
          .drop_duplicates()
          .sort_values(["GeneSymbol", "Reactome_ID"])
)

print(merged.head())

merged.to_csv(r"/content/drive/My Drive/KG_Ebola/gene_entrez_reactome_pathways.csv", index=False)


     GeneSymbol EntrezID    Reactome_ID  \
9230      ABCB1     5243  R-HSA-2161517   
9231      ABCB1     5243  R-HSA-2161522   
9232      ABCB1     5243   R-HSA-382551   
9233      ABCB1     5243   R-HSA-382556   
9234      ABCB1     5243  R-HSA-9748784   

                                 PathwayName Evidence  \
9230        Abacavir transmembrane transport      IEA   
9231                           Abacavir ADME      IEA   
9232            Transport of small molecules      TAS   
9233  ABC-family proteins mediated transport      TAS   
9234                               Drug ADME      IEA   

                                                    URL  
9230  https://reactome.org/PathwayBrowser/#/R-HSA-21...  
9231  https://reactome.org/PathwayBrowser/#/R-HSA-21...  
9232  https://reactome.org/PathwayBrowser/#/R-HSA-38...  
9233  https://reactome.org/PathwayBrowser/#/R-HSA-38...  
9234  https://reactome.org/PathwayBrowser/#/R-HSA-97...  


Preprocess gene_entrez_reactome_pathways.csv file. only keep triplets

In [ ]:
gene_reactome = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/gene_entrez_reactome_pathways.csv")
#print(gene_reactome.head())

gene_reactome = gene_reactome.drop(columns=["EntrezID", "Reactome_ID", "Evidence", "URL"])

#print(gene_reactome.head())

gene_reactome["Relation"] = "involved_in(Reactome)"

gene_reactome = gene_reactome[["GeneSymbol", "Relation", "PathwayName"]]

#print(gene_reactome.head())
gene_reactome.to_csv(Path("/content/drive/My Drive/KG_Ebola/protein_pathway_reactome.csv"), index=False)

Merge all csv file together to get a single .csv file

In [ ]:
ppi_df = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/ppi.csv")
print("PPI")
print(ppi_df.shape)
drug_target = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/drug_target_actions_final.csv")
print("drug target")
print(drug_target.shape)
drug_pathway = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/drug_pathway_simple.csv")
print("drug pathway")
print(drug_pathway.shape)
prot_path = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/protein_pathway_edges_filtered.csv")
print("protein pathway")
print(prot_path.shape)
gene_reactome = pd.read_csv(r"/content/drive/My Drive/KG_Ebola/protein_pathway_reactome.csv")
print("gene pathway")
print(gene_reactome.shape)


PPI
(3812, 3)
drug target
(557, 3)
drug pathway
(349, 3)
protein pathway
(5823, 3)
gene pathway
(16673, 3)


In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/content/drive/My Drive/KG_Ebola")

# 1. Load the files
ppi_df        = pd.read_csv(root / "ppi.csv")
drug_target   = pd.read_csv(root / "drug_target_actions_final.csv")
drug_pathway  = pd.read_csv(root / "drug_pathway_simple.csv")
prot_path     = pd.read_csv(root / "protein_pathway_edges_filtered.csv")
gene_reactome = pd.read_csv(root / "protein_pathway_reactome.csv")

# 2. Re‑map columns for each DataFrame
ppi_df = (ppi_df
          .rename(columns={"Ebola_protein": "Source",
                           "Human_protein": "Target"})
          [["Source", "Relation", "Target"]]).drop_duplicates()

drug_target = (drug_target
               .rename(columns={"DrugName": "Source",
                                "GeneSymbol": "Target"})
               [["Source", "Relation", "Target"]]).drop_duplicates()

drug_pathway = (drug_pathway
                .rename(columns={"DrugName": "Source",
                                 "PathwayCategory": "Relation",
                                 "PathwayName": "Target"})
                [["Source", "Relation", "Target"]]).drop_duplicates()

prot_path = (prot_path
             .rename(columns={"GeneName": "Source",
                              "relation": "Relation",
                              "PathwayName": "Target"})
             [["Source", "Relation", "Target"]]).drop_duplicates()

gene_reactome = (gene_reactome
                 .rename(columns={"GeneSymbol": "Source",
                                  "PathwayName": "Target"})
                 [["Source", "Relation", "Target"]]).drop_duplicates()

# 3. Save each individual SRT file (optional but handy)
ppi_df.to_csv(root / "ppi_df_SRT.csv", index=False)
drug_target.to_csv(root / "drug_target_SRT.csv", index=False)
drug_pathway.to_csv(root / "drug_pathway_SRT.csv", index=False)
prot_path.to_csv(root / "prot_path_SRT.csv", index=False)
gene_reactome.to_csv(root / "gene_reactome_SRT.csv", index=False)

# 4. Concatenate all into one master edge list
all_edges = (
    pd.concat([ppi_df, drug_target, drug_pathway, prot_path, gene_reactome],
              ignore_index=True)
      .drop_duplicates()                 # remove exact duplicates
      .replace('', pd.NA)                # treat empty strings as missing
      .dropna(how="any")                 # drop rows that have any NA
)

all_edges.to_csv(root / "Ebola_KG_all.csv", index=False)
print(f"✅ Master edge list saved → Ebola_KG_all.csv  ({len(all_edges):,} rows)")


✅ Master edge list saved → Ebola_KG_all.csv  (20,291 rows)


In [ ]:
# --- Node sets ---

# Ebola proteins
ebola_nodes = set(ppi_df["Source"].dropna().unique())

# Human proteins
human_nodes = (
    set(ppi_df["Target"].dropna().unique())
    | set(drug_target["Target"].dropna().unique())
    | set(prot_path["Source"].dropna().unique())
    | set(gene_reactome["Source"].dropna().unique())
)

# Drugs
drug_nodes = (
    set(drug_target["Source"].dropna().unique())
    | set(drug_pathway["Source"].dropna().unique())
)

# Pathways (DrugBank) – combine drug_pathway and prot_path Targets
drugbank_pathways = (
    set(drug_pathway["Target"].dropna().unique())
    | set(prot_path["Target"].dropna().unique())
)

# Reactome pathways
reactome_pathways = set(gene_reactome["Target"].dropna().unique())

# Merge DrugBank and Reactome for total pathways
all_pathways = drugbank_pathways | reactome_pathways

# All unique nodes
all_nodes = ebola_nodes | human_nodes | drug_nodes | all_pathways

# --- Print counts ---
print("Unique node counts:")
print(f"  Ebola proteins: {len(ebola_nodes)}")
print(f"  Human proteins: {len(human_nodes)}")
print(f"  Drugs: {len(drug_nodes)}")
print(f"  Pathways (DrugBank merged): {len(drugbank_pathways)}")
print(f"  Pathways (Reactome): {len(reactome_pathways)}")
print(f"  Total unique pathways: {len(all_pathways)}")
print(f"  Total unique nodes (all types): {len(all_nodes)}")


Unique node counts:
  Ebola proteins: 7
  Human proteins: 797
  Drugs: 293
  Pathways (DrugBank merged): 531
  Pathways (Reactome): 1763
  Total unique pathways: 2292
  Total unique nodes (all types): 3389
